In [1]:
# importing the required libraries
import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras import layers 
from tensorflow.keras.datasets import imdb 
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# loading the inbuilt imdb dataset 

(x_train,y_train), (x_test,y_test) = imdb.load_data(
    num_words= 10000
)

In [3]:
# lets choose max lenght of 200 for padding

max_length = 200 

x_train = pad_sequences(
    x_train,
    maxlen= max_length
)

x_test = pad_sequences(
    x_test,
    maxlen= max_length
)

In [4]:
# Build stacked GRU model
with tf.device('/GPU:0'):
    
    model = keras.Sequential([
        layers.Input(shape=(200,)),

        layers.Embedding(
            input_dim=10000,
            output_dim=128
        ),

        layers.GRU(
            64,
            return_sequences=True
        ),

        layers.GRU(32),

        layers.Dense(
            1,
            activation='sigmoid'
        )
    ])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 200, 128)          1280000   
                                                                 
 gru (GRU)                   (None, 200, 64)           37248     
                                                                 
 gru_1 (GRU)                 (None, 32)                9408      
                                                                 
 dense (Dense)               (None, 1)                 33        
                                                                 
Total params: 1,326,689
Trainable params: 1,326,689
Non-trainable params: 0
_________________________________________________________________


In [5]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=5,
    validation_split=0.2,
    callbacks=[early_stopping]
)

Epoch 1/5
313/313 [==============================] - 18s 40ms/step - loss: 0.4379 - accuracy: 0.7836 - val_loss: 0.3437 - val_accuracy: 0.8566
Epoch 2/5
313/313 [==============================] - 12s 39ms/step - loss: 0.2345 - accuracy: 0.9100 - val_loss: 0.3517 - val_accuracy: 0.8512
Epoch 3/5
313/313 [==============================] - 12s 37ms/step - loss: 0.1592 - accuracy: 0.9428 - val_loss: 0.3535 - val_accuracy: 0.8648


In [6]:
test_loss,test_accuracy = model.evaluate(
    x_test,
    y_test,
    batch_size=64
)

print('Test_Loss:     ',test_loss)
print('Test_Accuracy: ',test_accuracy)

391/391 [==============================] - 7s 19ms/step - loss: 0.3458 - accuracy: 0.8537
Test_Loss:      0.3457893431186676
Test_Accuracy:  0.8537200093269348


In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

texts = [
    "This movie was fantastic and I loved every minute of it.",
    "Absolutely terrible, a complete waste of time and money.",
    "A decent film, but it could have been much better."
]

max_features = 10000
maxlen = 200

# Get IMDB vocabulary
word_index = imdb.get_word_index()

# Create tokenizer
tokenizer = Tokenizer(
    num_words=max_features,
    oov_token="<OOV>"
)

# Use IMDB vocabulary
tokenizer.word_index = {
    word: index + 3
    for word, index in word_index.items()
}

tokenizer.word_index["<OOV>"] = 2

# Create reverse word index
tokenizer.index_word = {
    index: word
    for word, index in tokenizer.word_index.items()
}

# Convert text into IMDB word IDs
sequences = tokenizer.texts_to_sequences(texts)

# Display encoded sequences
for text, sequence in zip(texts, sequences):
    print("Original:", text)
    print("Encoded :", sequence)
    print()

# Pad sequences
padded_sequences = pad_sequences(
    sequences,
    maxlen=maxlen
)

print("Padded shape:", padded_sequences.shape)

# Predict
predictions = model.predict(
    padded_sequences,
    verbose=0
)

# Class names
class_names = ["Negative", "Positive"]

# Display predictions
for text, prediction in zip(texts, predictions):

    probability = prediction[0]

    predicted_class = 1 if probability >= 0.5 else 0

    confidence = probability if predicted_class == 1 else 1 - probability

    print("Text:", text)
    print("Predicted Sentiment:", class_names[predicted_class])
    print("Predicted Class:", predicted_class)
    print("Confidence:", f"{confidence:.4f}")
    print()

Original: This movie was fantastic and I loved every minute of it.
Encoded : [14, 20, 16, 777, 5, 13, 447, 175, 786, 7, 12]

Original: Absolutely terrible, a complete waste of time and money.
Encoded : [427, 394, 6, 601, 437, 7, 58, 5, 278]

Original: A decent film, but it could have been much better.
Encoded : [6, 542, 22, 21, 12, 100, 28, 77, 76, 128]

Padded shape: (3, 200)
Text: This movie was fantastic and I loved every minute of it.
Predicted Sentiment: Positive
Predicted Class: 1
Confidence: 0.8925

Text: Absolutely terrible, a complete waste of time and money.
Predicted Sentiment: Negative
Predicted Class: 0
Confidence: 0.9198

Text: A decent film, but it could have been much better.
Predicted Sentiment: Positive
Predicted Class: 1
Confidence: 0.5010

